# Day 1 PM — Claude Agent SDK live demonstrations

Run this notebook top-to-bottom. Every SDK cell uses the live Claude Agent SDK and Anthropic, with real teaching fixtures rather than invented records. If the SDK CLI runtime or `ANTHROPIC_API_KEY` is unavailable, the setup and runs fail clearly.

Two **Exercises** appear near the end (before Next). Attempt them before opening [`_EXERCISES_SOLUTIONS.ipynb`](_EXERCISES_SOLUTIONS.ipynb). On Windows Jupyter, live SDK cells use `run_sdk(...)`.

The implementation in `Week2/W2D1/app/README.md` is study-only context; this notebook is the executable lesson. Read-only cells run normally. A hold release is a write and requires explicit learner approval plus an idempotency key.

## 01 The Claude Agent SDK

The Claude Agent SDK exposes the same agent loop as Claude Code as a Python library. Use one-shot `query()` for bounded work; use `ClaudeSDKClient` when the next turn needs the same session. Inspect `ResultMessage` rather than guessing about session, turns, or cost.

<img src="Images/d1pm_t1_claude_agent_sdk.png" width="850" alt="Claude Agent SDK bounded query or client session with governed tools">

Run the shared setup, then the one-shot and multi-turn cells. The setup uses `cwd=W2D1PM` and `setting_sources=["project"]`, so this folder's `CLAUDE.md` and `AGENTS.md` become project context.

In [5]:
print()

In [6]:
import asyncio
import json
import os
import sys
from collections.abc import Callable, Coroutine
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any, Literal, TypeVar

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    SystemMessage,
    TextBlock,
    ThinkingBlock,
    ToolResultBlock,
    ToolUseBlock,
    query,
)
from dotenv import load_dotenv
from pydantic import BaseModel, Field


def discover_week2_root() -> Path:
    """Find Week2 by walking upward from the kernel working directory."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "W2D1" / "W2D1PM").is_dir() and (
            candidate / "requirements.txt"
        ).is_file():
            return candidate
    raise RuntimeError("Start the notebook from this repository or one of its subdirectories.")


WEEK2_ROOT = discover_week2_root()
NOTEBOOK_DIR = WEEK2_ROOT / "W2D1" / "W2D1PM"
FNOL_PATH = WEEK2_ROOT / "data" / "insurance" / "fnol_emails.csv"
CLAIMS_DB = WEEK2_ROOT / "data" / "insurance" / "claims.db"
SHIPMENTS_PATH = WEEK2_ROOT / "data" / "logistics" / "shipment_events.jsonl"
OUTPUT_DIR = WEEK2_ROOT / "W2D1" / "outputs" / "day1"
LEDGER_PATH = OUTPUT_DIR / "hold_release_ledger.jsonl"

load_dotenv(WEEK2_ROOT / ".env")

T = TypeVar("T")


def run_sdk(factory: Callable[[], Coroutine[Any, Any, T]]) -> T:
    """Run Claude Agent SDK work on a loop that can spawn Claude Code.

    On Windows, Jupyter often uses SelectorEventLoop, which cannot create
    subprocesses. The SDK needs ProactorEventLoop for the Claude Code CLI.
    """

    def _runner() -> T:
        loop = (
            asyncio.ProactorEventLoop()
            if sys.platform == "win32"
            else asyncio.new_event_loop()
        )
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(factory())
        finally:
            loop.run_until_complete(loop.shutdown_asyncgens())
            loop.close()

    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(_runner).result()


# Independent safety controls. Read-only cells do not consult these flags.
ALLOW_LOCAL_WRITES = False
ALLOW_HOLD_RELEASE = False
LEARNER_APPROVED_HOLD_RELEASE = False
HOLD_RELEASE_IDEMPOTENCY_KEY = ""


def require_settings() -> tuple[str, int, float, float]:
    """Require only the configuration shared by every SDK demonstration."""
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("Set ANTHROPIC_API_KEY in Week2/.env before running Day 1 PM.")
    return (
        os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
        int(os.getenv("AGENT_MAX_STEPS", "10")),
        float(os.getenv("AGENT_MAX_BUDGET_USD", "0.50")),
        float(os.getenv("AGENT_REQUEST_TIMEOUT_SECONDS", "60")),
    )


MODEL, MAX_TURNS, MAX_BUDGET_USD, TIMEOUT_SECONDS = require_settings()
SOURCE_AVAILABILITY = {
    "fnol_csv": {"path": str(FNOL_PATH), "available": FNOL_PATH.is_file()},
    "claims_db": {"path": str(CLAIMS_DB), "available": CLAIMS_DB.is_file()},
    "shipment_events": {
        "path": str(SHIPMENTS_PATH),
        "available": SHIPMENTS_PATH.is_file(),
    },
}
print(
    {
        "week2_root": str(WEEK2_ROOT),
        "model": MODEL,
        "max_turns": MAX_TURNS,
        "max_budget_usd": MAX_BUDGET_USD,
        "timeout_seconds": TIMEOUT_SECONDS,
        "data_sources": SOURCE_AVAILABILITY,
        "local_writes_enabled": ALLOW_LOCAL_WRITES,
        "hold_release_enabled": ALLOW_HOLD_RELEASE,
    }
)

{'week2_root': 'D:\\Ambilio\\EXL-CampusHire\\Week2', 'model': 'claude-sonnet-4-6', 'max_turns': 10, 'max_budget_usd': 0.5, 'timeout_seconds': 60.0, 'data_sources': {'fnol_csv': {'path': 'D:\\Ambilio\\EXL-CampusHire\\Week2\\data\\insurance\\fnol_emails.csv', 'available': True}, 'claims_db': {'path': 'D:\\Ambilio\\EXL-CampusHire\\Week2\\data\\insurance\\claims.db', 'available': True}, 'shipment_events': {'path': 'D:\\Ambilio\\EXL-CampusHire\\Week2\\data\\logistics\\shipment_events.jsonl', 'available': True}}, 'local_writes_enabled': False, 'hold_release_enabled': False}


In [7]:
options = ClaudeAgentOptions(
    model=MODEL,
    tools=[],
    max_turns=MAX_TURNS,
    max_budget_usd=MAX_BUDGET_USD,
    cwd=NOTEBOOK_DIR,
    setting_sources=["project"],
)


async def _sdk_run() -> ResultMessage:
    result: ResultMessage | None = None
    async for message in query(
        prompt=(
            "In two concise sentences, explain why an insurance agent must use "
            "domain tools instead of inventing claim facts."
        ),
        options=options,
    ):
        if isinstance(message, ResultMessage):
            result = message

    if result is None:
        raise RuntimeError("SDK query ended without a ResultMessage.")
    if result.is_error:
        raise RuntimeError(f"SDK run failed: {result.subtype}")
    return result


result = run_sdk(_sdk_run)
print(result.result)
print(
    {
        "session_id": result.session_id,
        "num_turns": result.num_turns,
        "stop_reason": result.stop_reason,
        "usage": result.usage,
        "total_cost_usd": result.total_cost_usd,
    }
)

Domain tools ensure that claim, policy, and FNOL data reflect authoritative system-of-record values, preventing inaccurate payouts, coverage disputes, or compliance violations that could arise from fabricated facts. Invented records also undermine the integrity of the `TriageDecision` output, exposing the insurer to fraud risk and regulatory liability.
{'session_id': '115faed6-f414-444b-9fe7-c3d4847fd6c3', 'num_turns': 1, 'stop_reason': 'end_turn', 'usage': {'input_tokens': 390, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 114, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'inference_geo': 'global', 'iterations': [{'input_tokens': 390, 'output_tokens': 114, 'cache_read_input_tokens': 0, 'cache_creation_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'type': 'mes

In [8]:
from claude_agent_sdk import ClaudeSDKClient


async def receive_result(client: ClaudeSDKClient) -> ResultMessage:
    async for message in client.receive_response():
        if isinstance(message, ResultMessage):
            return message
    raise RuntimeError("SDK client ended without a ResultMessage.")


async def _sdk_run() -> tuple[ResultMessage, ResultMessage]:
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Remember the rule: do not infer unknown claim records.")
        first = await receive_result(client)
        await client.query("What rule did I ask you to remember?")
        second = await receive_result(client)
    return first, second


first, second = run_sdk(_sdk_run)
if first.is_error or second.is_error:
    raise RuntimeError(f"SDK client failed: {first.subtype} / {second.subtype}")
print(second.result)
print({"session_id": second.session_id, "num_turns": second.num_turns, "total_cost_usd": second.total_cost_usd})

You asked me to remember:

> **Do not infer unknown claim records.**

This aligns with the project rule in CLAUDE.md: never invent claim, loan, policy, or shipment records — if a domain tool returns no result, report that clearly rather than fabricating data.
{'session_id': '8b27d600-a69f-43a6-9857-27a234a2344f', 'num_turns': 1, 'total_cost_usd': 0.006742}


In [9]:
def describe_block(block: object) -> dict[str, object]:
    """Expose native artifacts without printing hidden thinking text."""
    if isinstance(block, TextBlock):
        return {"type": "text", "text": block.text}
    if isinstance(block, ThinkingBlock):
        return {"type": "thinking", "characters": len(block.thinking)}
    if isinstance(block, ToolUseBlock):
        return {"type": "tool_use", "name": block.name, "input": block.input}
    if isinstance(block, ToolResultBlock):
        return {
            "type": "tool_result",
            "tool_use_id": block.tool_use_id,
            "is_error": block.is_error,
        }
    return {"type": type(block).__name__}


async def _sdk_run() -> list[dict[str, object]]:
    stream_evidence: list[dict[str, object]] = []
    async for message in query(
        prompt="Give a three-item operator checklist for grounded FNOL triage.",
        options=options,
    ):
        if isinstance(message, SystemMessage):
            stream_evidence.append(
                {"message": "system", "subtype": message.subtype, "data": message.data}
            )
        elif isinstance(message, AssistantMessage):
            stream_evidence.append(
                {
                    "message": "assistant",
                    "model": message.model,
                    "blocks": [describe_block(block) for block in message.content],
                    "usage": message.usage,
                    "stop_reason": message.stop_reason,
                }
            )
        elif isinstance(message, ResultMessage):
            stream_evidence.append(
                {
                    "message": "result",
                    "subtype": message.subtype,
                    "is_error": message.is_error,
                    "session_id": message.session_id,
                    "num_turns": message.num_turns,
                    "stop_reason": message.stop_reason,
                    "usage": message.usage,
                    "total_cost_usd": message.total_cost_usd,
                }
            )
    return stream_evidence


stream_evidence = run_sdk(_sdk_run)
print(json.dumps(stream_evidence, default=str, indent=2))

[
  {
    "message": "system",
    "subtype": "init",
    "data": {
      "type": "system",
      "subtype": "init",
      "cwd": "D:\\Ambilio\\EXL-CampusHire\\Week2\\W2D1\\W2D1PM",
      "session_id": "f7f5d3b2-44c1-4ae3-8f66-dcc926853ab6",
      "tools": [],
      "mcp_servers": [],
      "model": "claude-sonnet-4-6",
      "permissionMode": "default",
      "slash_commands": [
        "deep-research",
        "design-sync",
        "dataviz",
        "update-config",
        "verify",
        "debug",
        "code-review",
        "simplify",
        "batch",
        "fewer-permission-prompts",
        "doctor",
        "loop",
        "claude-api",
        "run",
        "run-skill-generator",
        "agents",
        "clear",
        "color",
        "compact",
        "config",
        "context",
        "effort",
        "fast",
        "heapdump",
        "init",
        "mcp",
        "model",
        "__remote-workflow",
        "reload-skills",
        "rename",
        "r

In [10]:
class TriageDecision(BaseModel):
    """Stable Week 2 decision contract."""

    lane: Literal["insurance", "banking", "logistics"]
    case_id: str
    urgency: Literal["low", "medium", "high"]
    route_queue: str
    summary: str
    rationale: str
    tools_used: list[str]
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_approval: bool = False


options = ClaudeAgentOptions(
    model=MODEL,
    tools=[],
    output_format={"type": "json_schema", "schema": TriageDecision.model_json_schema()},
    max_turns=MAX_TURNS,
    max_budget_usd=MAX_BUDGET_USD,
    cwd=NOTEBOOK_DIR,
    setting_sources=["project"],
)


async def _sdk_run() -> ResultMessage:
    result: ResultMessage | None = None
    async for message in query(
        prompt=(
            "Return a schema-only TriageDecision for known fixture CLN-001. Set lane=insurance, "
            "urgency=low, route_queue=training_review, tools_used=[], confidence=0.5, "
            "and needs_human_approval=true. State that factual triage requires the domain tools."
        ),
        options=options,
    ):
        if isinstance(message, ResultMessage):
            result = message

    if result is None or result.is_error:
        raise RuntimeError("Structured SDK run did not finish successfully.")
    return result


result = run_sdk(_sdk_run)
decision = TriageDecision.model_validate(result.structured_output)
print(decision.model_dump_json(indent=2))
print(
    json.dumps(
        {
            "session_id": result.session_id,
            "subtype": result.subtype,
            "usage": result.usage,
            "cost_usd": result.total_cost_usd,
        },
        default=str,
        indent=2,
    )
)

{
  "lane": "insurance",
  "case_id": "CLN-001",
  "urgency": "low",
  "route_queue": "training_review",
  "summary": "Schema-only TriageDecision for fixture CLN-001. No domain tools were invoked; all fields are populated from caller-supplied values for training/validation purposes only.",
  "rationale": "Factual triage — including policy lookup, FNOL validation, and claim status checks — requires invocation of the designated insurance domain tools. Because no tools were called for this case, the resulting decision carries no evidentiary basis and must not be treated as an operational routing outcome. Confidence is therefore set to 0.5 (coin-flip) and needs_human_approval is flagged true so a reviewer can validate or reject before any downstream action is taken.",
  "tools_used": [],
  "confidence": 0.5,
  "needs_human_approval": true
}
{
  "session_id": "00c56d1e-ec7d-4dc4-99a6-e3ce27975580",
  "subtype": "success",
  "usage": {
    "input_tokens": 3,
    "cache_creation_input_tokens"

## 02 Built-in & custom tools

`tools=[]` gives the agent no built-in tools. `allowed_tools` allows only named tools; built-ins such as `Read` and `Bash` are powerful and should be deliberately scoped. For the small classroom read, the agent may read local project instructions. The main example uses `@tool` plus `create_sdk_mcp_server` to expose authoritative FNOL, claim, and policy lookups as `mcp__ops__...` tools.

<img src="Images/d1pm_t2_builtin_custom_tools.png" width="850" alt="Built-in tools versus governed custom MCP domain tools">

Tool results are the evidence; the model must not invent records.

In [16]:
import csv
import sqlite3

from claude_agent_sdk import ToolAnnotations, UserMessage, create_sdk_mcp_server, tool
from claude_agent_sdk.types import (
    PermissionResultAllow,
    PermissionResultDeny,
    ToolPermissionContext,
)


missing_insurance_sources = [path for path in (FNOL_PATH, CLAIMS_DB) if not path.is_file()]
if missing_insurance_sources:
    raise FileNotFoundError(
        "Topic 2 insurance tools require fnol_emails.csv and claims.db. "
        f"Missing: {[str(path) for path in missing_insurance_sources]}. "
        "Restore the FNOL teaching fixture if absent; generate claims.db with "
        "`uv run python data/insurance/seed_claims_db.py` from Week2."
    )


def text_result(payload: object, *, is_error: bool = False) -> dict[str, Any]:
    return {
        "content": [{"type": "text", "text": json.dumps(payload)}],
        "is_error": is_error,
    }


@tool(
    "fnol_lookup",
    "Return one authoritative FNOL email record by CLN identifier.",
    {
        "type": "object",
        "properties": {"email_id": {"type": "string", "pattern": "^CLN-[0-9]{3}$"}},
        "required": ["email_id"],
        "additionalProperties": False,
    },
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def fnol_lookup(args: dict[str, Any]) -> dict[str, Any]:
    with FNOL_PATH.open(encoding="utf-8", newline="") as handle:
        row = next(
            (item for item in csv.DictReader(handle) if item["email_id"] == args["email_id"]),
            None,
        )
    return text_result(row or {"found": False, "email_id": args["email_id"]})


def database_result(table: str, column: str, identifier: str) -> dict[str, Any]:
    if (table, column) not in {("claims", "claim_id"), ("policies", "policy_id")}:
        return text_result({"error": "query not allowlisted"}, is_error=True)
    with sqlite3.connect(CLAIMS_DB) as connection:
        connection.row_factory = sqlite3.Row
        row = connection.execute(
            f"SELECT * FROM {table} WHERE {column} = ?", (identifier,)
        ).fetchone()
    return text_result(dict(row) if row else {"found": False, "id": identifier})


@tool(
    "claim_lookup",
    "Find one authoritative claim by CLM identifier.",
    {"claim_id": str},
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def claim_lookup(args: dict[str, Any]) -> dict[str, Any]:
    return database_result("claims", "claim_id", args["claim_id"])


@tool(
    "policy_lookup",
    "Find one authoritative policy by POL identifier.",
    {"policy_id": str},
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def policy_lookup(args: dict[str, Any]) -> dict[str, Any]:
    return database_result("policies", "policy_id", args["policy_id"])


OPS_TOOL_NAMES = [
    "mcp__ops__fnol_lookup",
    "mcp__ops__claim_lookup",
    "mcp__ops__policy_lookup",
]
ops_server = create_sdk_mcp_server(
    name="ops",
    version="1.0.0",
    tools=[fnol_lookup, claim_lookup, policy_lookup],
)

# MCP tool names use the mcp_servers dict key: mcp__<key>__<tool>.
INSURANCE_TOOL_NAMES = [
    "mcp__insurance__fnol_lookup",
    "mcp__insurance__claim_lookup",
    "mcp__insurance__policy_lookup",
]
insurance_server = create_sdk_mcp_server(
    name="insurance",
    version="1.0.0",
    tools=[fnol_lookup, claim_lookup, policy_lookup],
)
print(
    {
        "ops_tools": OPS_TOOL_NAMES,
        "insurance_tools": INSURANCE_TOOL_NAMES,
    }
)

{'ops_tools': ['mcp__ops__fnol_lookup', 'mcp__ops__claim_lookup', 'mcp__ops__policy_lookup'], 'insurance_tools': ['mcp__insurance__fnol_lookup', 'mcp__insurance__claim_lookup', 'mcp__insurance__policy_lookup']}


In [17]:
read_options = ClaudeAgentOptions(
    model=MODEL,
    tools=["Read"],
    allowed_tools=["Read"],
    max_turns=3,
    max_budget_usd=0.10,
    cwd=NOTEBOOK_DIR,
    setting_sources=["project"],
)


async def _sdk_run() -> ResultMessage:
    read_result: ResultMessage | None = None
    async for message in query(
        prompt="Use Read to inspect CLAUDE.md, then state its first operator rule.",
        options=read_options,
    ):
        if isinstance(message, ResultMessage):
            read_result = message
    if read_result is None or read_result.is_error:
        raise RuntimeError(
            "Built-in Read demo failed; check the SDK CLI runtime and Anthropic access."
        )
    return read_result


read_result = run_sdk(_sdk_run)
print(read_result.result)
print("Bash is intentionally not enabled in this classroom example.")

The file was read successfully. Here is a summary of what was found:

---

**📄 CLAUDE.md — First Operator Rule:**

> **"Use domain tools for factual data."**

This is the very first rule listed under the **Rules** section (line 11). It mandates that all factual information — such as claims, loans, policies, or shipments — must be retrieved through the appropriate domain tools, rather than assumed or fabricated. This directly underpins the second rule: *"Never invent claim, loan, policy, or shipment records."*
Bash is intentionally not enabled in this classroom example.


In [18]:
# `tools=[]` disables built-ins. To demonstrate a safe built-in, replace it with
# `tools=["Read"]`, `allowed_tools=["Read"]`, and prompt only for CLAUDE.md.
fnol_only_server = create_sdk_mcp_server(
    name="ops",
    version="1.0.0",
    tools=[fnol_lookup],
)
options = ClaudeAgentOptions(
    model=MODEL,
    tools=[],
    mcp_servers={"ops": fnol_only_server},
    allowed_tools=["mcp__ops__fnol_lookup"],
    max_turns=MAX_TURNS,
    max_budget_usd=MAX_BUDGET_USD,
    cwd=NOTEBOOK_DIR,
    setting_sources=["project"],
)


async def _sdk_run() -> tuple[list[dict[str, object]], ResultMessage]:
    tool_calls: list[dict[str, object]] = []
    result: ResultMessage | None = None
    async for message in query(
        prompt="Call fnol_lookup for CLN-001, then summarize only the returned record.",
        options=options,
    ):
        if isinstance(message, AssistantMessage):
            tool_calls.extend(
                {"tool": block.name, "input": block.input}
                for block in message.content
                if isinstance(block, ToolUseBlock)
            )
        elif isinstance(message, ResultMessage):
            result = message

    if result is None or result.is_error:
        raise RuntimeError("FNOL tool demonstration did not finish successfully.")
    return tool_calls, result


tool_calls, result = run_sdk(_sdk_run)
print(json.dumps({"tool_calls": tool_calls}, indent=2))
print(result.result)

{
  "tool_calls": [
    {
      "tool": "mcp__ops__fnol_lookup",
      "input": {
        "email_id": "CLN-001"
      }
    }
  ]
}
Here is a concise summary of the **CLN-001** record:

| Field | Value |
|---|---|
| **Email ID** | CLN-001 |
| **Category** | FNOL |
| **Claim #** | CLM-424063 |
| **Policy #** | POL-787532354 |
| **Claimant** | Jennifer Garcia |
| **Loss Location** | 6650 Broadway |
| **Loss Type** | Vandalism / Fence Damage |
| **Estimated Damage** | $5,834 |
| **Police Report** | PR-20250822-4864 |
| **Supporting Docs** | Photos available |
| **Urgency** | 🔴 High |
| **Route Queue** | `fnol_intake` |

**Operator Note:** This is a high-urgency FNOL submission. The claimant has photos and a police report on file. Recommend prioritizing intake and verifying coverage under policy **POL-787532354** promptly.


In [19]:
options = ClaudeAgentOptions(
    model=MODEL,
    tools=[],
    mcp_servers={"insurance": insurance_server},
    allowed_tools=INSURANCE_TOOL_NAMES,
    output_format={"type": "json_schema", "schema": TriageDecision.model_json_schema()},
    max_turns=MAX_TURNS,
    max_budget_usd=MAX_BUDGET_USD,
    cwd=NOTEBOOK_DIR,
    setting_sources=["project"],
)


async def _sdk_run() -> tuple[list[str], list[dict[str, object]], ResultMessage]:
    called_tools: list[str] = []
    tool_results: list[dict[str, object]] = []
    result: ResultMessage | None = None
    async for message in query(
        prompt=(
            "Triage CLN-001. Call fnol_lookup first, then claim_lookup and policy_lookup "
            "using IDs from that record. Return TriageDecision using exact evidence values "
            "and use the CLM claim ID as case_id."
        ),
        options=options,
    ):
        if isinstance(message, AssistantMessage):
            called_tools.extend(
                block.name for block in message.content if isinstance(block, ToolUseBlock)
            )
        elif isinstance(message, UserMessage):
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    tool_results.append(
                        {
                            "tool_use_id": block.tool_use_id,
                            "content": block.content,
                            "is_error": block.is_error,
                        }
                    )
        elif isinstance(message, ResultMessage):
            result = message

    if result is None or result.is_error:
        raise RuntimeError("Grounded triage did not finish successfully.")
    return called_tools, tool_results, result


called_tools, tool_results, result = run_sdk(_sdk_run)
decision = TriageDecision.model_validate(result.structured_output)
required_tools = set(INSURANCE_TOOL_NAMES)
if not required_tools.issubset(called_tools):
    raise ValueError(f"Missing required tool evidence: {required_tools - set(called_tools)}")
evidence_text = json.dumps(tool_results, default=str)
if not decision.case_id.startswith("CLM-") or decision.case_id not in evidence_text:
    raise ValueError("case_id must be the evidence-backed claim ID.")
if any(item["is_error"] for item in tool_results):
    raise ValueError("Grounded triage cannot accept errored tool evidence.")

print(decision.model_dump_json(indent=2))
print(
    json.dumps(
        {
            "called_tools": called_tools,
            "tool_results": tool_results,
            "session_id": result.session_id,
            "usage": result.usage,
            "cost_usd": result.total_cost_usd,
        },
        default=str,
        indent=2,
    )
)

{
  "lane": "insurance",
  "case_id": "CLM-424063",
  "urgency": "high",
  "route_queue": "fnol_intake",
  "summary": "Named insured Jennifer Garcia reports vandalism-related fence damage at 6650 Broadway. Estimated loss $5,834. Police report PR-20250822-4864 filed. Photos available. Claim CLM-424063 is open against active homeowners policy POL-787532354. Reserve matches estimated damage exactly.",
  "rationale": "FNOL CLN-001 identifies claim CLM-424063 and policy POL-787532354. Claim record confirms: status=open, loss_type=property, urgency=high, reserve_usd=$5,834, route_queue=fnol_intake. Policy record confirms: named_insured=Jennifer Garcia, status=active, product=homeowners. All three records are internally consistent — claimant name, policy ID, and reserve amount align across FNOL, claim, and policy. No write operations requested; no human approval required.",
  "tools_used": [
    "mcp__insurance__fnol_lookup",
    "mcp__insurance__claim_lookup",
    "mcp__insurance__policy_loo

## 03 Structured outputs & message blocks

SDK streaming yields native message and block types: `TextBlock`, `ToolUseBlock`, and `ToolResultBlock`; some SDK versions also expose `ThinkingBlock`. The next run records block type names and validates the SDK JSON-schema result with `TriageDecision.model_validate()`.

<img src="Images/d1pm_t3_structured_outputs.png" width="850" alt="Message blocks flowing into a grounded TriageDecision">

In [20]:
READ_TOOLS = {*INSURANCE_TOOL_NAMES, "StructuredOutput"}
permission_log: list[dict[str, object]] = []


async def read_permission_gate(
    tool_name: str,
    input_data: dict[str, Any],
    context: ToolPermissionContext,
) -> PermissionResultAllow | PermissionResultDeny:
    allowed = tool_name in READ_TOOLS
    permission_log.append({"tool": tool_name, "input": input_data, "allowed": allowed})
    if allowed:
        return PermissionResultAllow(updated_input=input_data)
    return PermissionResultDeny(message="Tool is outside the Day 1 read-only policy.")


async def prompt_stream(text: str):
    yield {"type": "user", "message": {"role": "user", "content": text}}


options = ClaudeAgentOptions(
    model=MODEL,
    tools=[],
    mcp_servers={"insurance": insurance_server},
    can_use_tool=read_permission_gate,
    output_format={"type": "json_schema", "schema": TriageDecision.model_json_schema()},
    max_turns=MAX_TURNS,
    max_budget_usd=MAX_BUDGET_USD,
    cwd=NOTEBOOK_DIR,
    setting_sources=["project"],
)


async def _sdk_run() -> ResultMessage:
    result: ResultMessage | None = None
    async for message in query(
        prompt=prompt_stream(
            "Triage CLN-001 using FNOL, claim, and policy tools. Return only evidence-backed "
            "fields in TriageDecision and use the CLM claim ID as case_id."
        ),
        options=options,
    ):
        if isinstance(message, ResultMessage):
            result = message

    if result is None or result.is_error:
        raise RuntimeError("Permission-gated triage did not finish successfully.")
    return result


result = run_sdk(_sdk_run)
decision = TriageDecision.model_validate(result.structured_output)
if not decision.case_id.startswith("CLM-"):
    raise ValueError("TriageDecision must use an evidence-backed claim ID.")
print(decision.model_dump_json(indent=2))
print(json.dumps({"permission_log": permission_log}, indent=2))

{
  "lane": "insurance",
  "case_id": "CLM-424063",
  "urgency": "high",
  "route_queue": "fnol_intake",
  "summary": "Jennifer Garcia (POL-787532354, active homeowners) filed FNOL CLN-001 reporting vandalism/fence damage at 6650 Broadway. Claim CLM-424063 is open with a $5,834 reserve matching the FNOL estimate. Police report PR-20250822-4864 and photos are on file. Route immediately to fnol_intake.",
  "rationale": "Three independent tool lookups are fully consistent: (1) FNOL CLN-001 identifies claimant Jennifer Garcia, claim CLM-424063, policy POL-787532354, loss type vandalism/property, estimated damage $5,834, and references police report PR-20250822-4864. (2) Claim record confirms CLM-424063 is open, loss_type=property, urgency=high, reserve_usd=$5,834, route_queue=fnol_intake. (3) Policy record confirms POL-787532354 is active, product=homeowners, named insured=Jennifer Garcia — matching the FNOL claimant exactly. No discrepancies detected; all fields are evidence-backed. No wr

In [21]:
if not SHIPMENTS_PATH.is_file():
    raise FileNotFoundError(
        "Shipment demonstrations require data/logistics/shipment_events.jsonl. "
        f"Restore the teaching fixture at {SHIPMENTS_PATH}, then rerun this cell."
    )

SHIPMENT_LOOKUP_TOOL = "mcp__logistics__shipment_lookup"
RELEASE_HOLD_TOOL = "mcp__logistics__release_hold"
PLACEHOLDER_KEYS = {"", "change-me", "changeme", "placeholder", "your-key-here"}


def shipment_records(container_id: str) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for line in SHIPMENTS_PATH.read_text(encoding="utf-8").splitlines():
        event = json.loads(line)
        if event["container_id"] == container_id:
            records.append(event)
    return records


def valid_idempotency_key(value: str) -> bool:
    normalized = value.strip().lower()
    return len(value.strip()) >= 12 and normalized not in PLACEHOLDER_KEYS


@tool(
    "shipment_lookup",
    "Return authoritative events for one shipment container.",
    {"container_id": str},
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def shipment_lookup(args: dict[str, Any]) -> dict[str, Any]:
    events = shipment_records(args["container_id"])
    return text_result(events or {"found": False, "container_id": args["container_id"]})


def record_hold_release(
    *,
    container_id: str,
    approved: bool,
    idempotency_key: str,
    source_event_id: str,
) -> dict[str, Any]:
    """Apply the deterministic write boundary; the model cannot override it."""
    if not ALLOW_LOCAL_WRITES:
        return {"executed": False, "reason": "local writes disabled"}
    if not ALLOW_HOLD_RELEASE:
        return {"executed": False, "reason": "hold release disabled"}
    if not approved or not LEARNER_APPROVED_HOLD_RELEASE:
        return {"executed": False, "reason": "learner approval required"}
    if not valid_idempotency_key(idempotency_key):
        return {"executed": False, "reason": "non-placeholder idempotency key required"}

    records = shipment_records(container_id)
    if not records:
        return {"executed": False, "reason": "shipment not found"}
    latest = records[-1]
    if latest["event_id"] != source_event_id:
        return {"executed": False, "reason": "proposal source event is stale or unsupported"}
    if latest.get("status") != "held":
        return {"executed": False, "reason": "latest authoritative status is not held"}

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    existing = (
        LEDGER_PATH.read_text(encoding="utf-8").splitlines() if LEDGER_PATH.exists() else []
    )
    prior = [json.loads(line) for line in existing]
    if any(item["idempotency_key"] == idempotency_key for item in prior):
        return {"executed": False, "duplicate": True, "idempotency_key": idempotency_key}

    command = {
        "action": "release_hold",
        "container_id": container_id,
        "idempotency_key": idempotency_key,
        "source_event_id": source_event_id,
        "executed": True,
    }
    with LEDGER_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(command) + "\n")
    return command


@tool(
    "release_hold",
    "Record an approved hold release; requires approval and an idempotency key.",
    {
        "container_id": str,
        "approved": bool,
        "idempotency_key": str,
        "source_event_id": str,
    },
    annotations=ToolAnnotations(readOnlyHint=False, destructiveHint=True),
)
async def release_hold(args: dict[str, Any]) -> dict[str, Any]:
    release_result = record_hold_release(
        container_id=args["container_id"],
        approved=args["approved"],
        idempotency_key=args["idempotency_key"],
        source_event_id=args["source_event_id"],
    )
    return text_result(release_result, is_error=not release_result.get("executed", False))


logistics_server = create_sdk_mcp_server(
    name="logistics",
    version="1.0.0",
    tools=[shipment_lookup, release_hold],
)
print(
    {
        "write_tool_registered": RELEASE_HOLD_TOOL,
        "local_writes_enabled": ALLOW_LOCAL_WRITES,
        "hold_release_enabled": ALLOW_HOLD_RELEASE,
        "learner_approved": LEARNER_APPROVED_HOLD_RELEASE,
        "idempotency_key_is_valid": valid_idempotency_key(HOLD_RELEASE_IDEMPOTENCY_KEY),
    }
)

{'write_tool_registered': 'mcp__logistics__release_hold', 'local_writes_enabled': False, 'hold_release_enabled': False, 'learner_approved': False, 'idempotency_key_is_valid': False}


In [22]:
class ReleaseProposal(BaseModel):
    """A typed recommendation that cannot authorize a write."""

    container_id: str
    source_event_id: str
    should_release: bool
    reason: str


release_permission_log: list[dict[str, object]] = []


async def release_permission_gate(
    tool_name: str,
    input_data: dict[str, Any],
    context: ToolPermissionContext,
) -> PermissionResultAllow | PermissionResultDeny:
    if tool_name == SHIPMENT_LOOKUP_TOOL:
        release_permission_log.append({"tool": tool_name, "allowed": True})
        return PermissionResultAllow(updated_input=input_data)

    write_allowed = (
        tool_name == RELEASE_HOLD_TOOL
        and ALLOW_LOCAL_WRITES
        and ALLOW_HOLD_RELEASE
        and LEARNER_APPROVED_HOLD_RELEASE
        and valid_idempotency_key(HOLD_RELEASE_IDEMPOTENCY_KEY)
        and input_data.get("idempotency_key") == HOLD_RELEASE_IDEMPOTENCY_KEY
    )
    release_permission_log.append({"tool": tool_name, "allowed": write_allowed})
    if write_allowed:
        return PermissionResultAllow(
            updated_input={
                **input_data,
                "approved": True,
                "idempotency_key": HOLD_RELEASE_IDEMPOTENCY_KEY,
            }
        )
    return PermissionResultDeny(
        message="Write requires both safety flags, learner approval, and the matching non-placeholder key."
    )


options = ClaudeAgentOptions(
    model=MODEL,
    tools=[],
    mcp_servers={"logistics": logistics_server},
    can_use_tool=release_permission_gate,
    max_turns=MAX_TURNS,
    max_budget_usd=MAX_BUDGET_USD,
    cwd=NOTEBOOK_DIR,
    setting_sources=["project"],
)


async def _sdk_run() -> tuple[list[dict[str, object]], ResultMessage]:
    tool_results: list[dict[str, object]] = []
    result: ResultMessage | None = None
    async for message in query(
        prompt=prompt_stream(
            "Call shipment_lookup for MSKU789. State whether the latest authoritative status "
            "is held. Do not call release_hold and do not claim that a release occurred."
        ),
        options=options,
    ):
        if isinstance(message, UserMessage):
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    tool_results.append(
                        {
                            "tool_use_id": block.tool_use_id,
                            "content": block.content,
                            "is_error": block.is_error,
                        }
                    )
        elif isinstance(message, ResultMessage):
            result = message

    if result is None or result.is_error:
        raise RuntimeError("Shipment preview did not finish successfully.")
    return tool_results, result


tool_results, result = run_sdk(_sdk_run)
shipment_evidence = shipment_records("MSKU789")
if not shipment_evidence:
    raise ValueError("Shipment not found; no release proposal can be created.")
latest_shipment = shipment_evidence[-1]
if latest_shipment["event_id"] not in json.dumps(tool_results):
    raise ValueError("Release proposal is unsupported by captured shipment tool evidence.")

release_preview = ReleaseProposal(
    container_id="MSKU789",
    source_event_id=latest_shipment["event_id"],
    should_release=latest_shipment["status"] == "held",
    reason=f"Latest authoritative shipment status is {latest_shipment['status']}.",
)
print(
    json.dumps(
        {
            "proposal_only": release_preview.model_dump(),
            "permission_log": release_permission_log,
            "sdk_result": result.result,
        },
        indent=2,
    )
)

{
  "proposal_only": {
    "container_id": "MSKU789",
    "source_event_id": "EVT-MSKU789-001",
    "should_release": true,
    "reason": "Latest authoritative shipment status is held."
  },
  "permission_log": [
    {
      "tool": "mcp__logistics__shipment_lookup",
      "allowed": true
    }
  ],
  "sdk_result": "Here are the findings for container **MSKU789**:\n\n| Field | Value |\n|---|---|\n| **Event ID** | EVT-MSKU789-001 |\n| **Timestamp** | 2026-07-12T16:45:00Z |\n| **Location** | Chicago Rail Yard |\n| **Status** | **HELD** |\n| **Detail** | Awaiting documentation \u2014 bill of lading mismatch |\n\n**Yes \u2014 the latest authoritative status is HELD.** The single event on record places the container at the Chicago Rail Yard as of 2026-07-12, flagged due to a bill of lading mismatch. No release has been recorded or initiated."
}


## 04 Memory & sessions

`ClaudeSDKClient` preserves conversation within a client; `resume=session_id` continues a stored session under the same project working directory. The next cell confirms that both `CLAUDE.md` and `AGENTS.md` exist where `cwd` points, then keeps one rule across two turns.

<img src="Images/d1pm_t4_memory_sessions.png" width="850" alt="Resumable sessions with summary, facts, approvals, and project context">

The SDK may compact context automatically when needed. Its `ResultMessage` does not guarantee a universal compaction field; inspect available result fields and SDK logs when observing compaction in a live run.

In [23]:
import time

from claude_agent_sdk import ClaudeSDKClient

# Paste a previously captured session ID to resume it under the same working directory.
RESUME_SESSION_ID: str | None = None


async def receive_result(client: ClaudeSDKClient) -> ResultMessage:
    final: ResultMessage | None = None
    async for message in client.receive_response():
        if isinstance(message, ResultMessage):
            final = message
    if final is None:
        raise RuntimeError("SDK response ended without a ResultMessage.")
    return final


print(
    {
        "cwd": str(NOTEBOOK_DIR),
        "CLAUDE.md_exists": (NOTEBOOK_DIR / "CLAUDE.md").is_file(),
        "AGENTS.md_exists": (NOTEBOOK_DIR / "AGENTS.md").is_file(),
    }
)
option_values: dict[str, Any] = {
    "model": MODEL,
    "tools": [],
    "max_turns": MAX_TURNS,
    "max_budget_usd": MAX_BUDGET_USD,
    "cwd": NOTEBOOK_DIR,
    "setting_sources": ["project"],
}
if RESUME_SESSION_ID:
    option_values["resume"] = RESUME_SESSION_ID

options = ClaudeAgentOptions(**option_values)


async def _sdk_run() -> tuple[ResultMessage, ResultMessage]:
    async with ClaudeSDKClient(options=options) as client:
        await client.query(
            "Remember this operating rule for our session: unknown claim records must "
            "be refused, not inferred. Confirm concisely."
        )
        first_result = await receive_result(client)
        await client.query("What operating rule did I ask you to remember?")
        second_result = await receive_result(client)
    return first_result, second_result


first_result, second_result = run_sdk(_sdk_run)
if first_result.session_id != second_result.session_id:
    raise RuntimeError("Expected both turns to use the same SDK session.")

session_artifact = {
    "session_id": second_result.session_id,
    "resumed_from": RESUME_SESSION_ID,
    "turn_1": first_result.result,
    "turn_2": second_result.result,
    "turn_1_usage": first_result.usage,
    "turn_2_usage": second_result.usage,
    "total_cost_usd": (
        (first_result.total_cost_usd or 0.0) + (second_result.total_cost_usd or 0.0)
    ),
}
print(json.dumps(session_artifact, default=str, indent=2))

{'cwd': 'D:\\Ambilio\\EXL-CampusHire\\Week2\\W2D1\\W2D1PM', 'CLAUDE.md_exists': True, 'AGENTS.md_exists': True}
{
  "session_id": "381ea71a-8f23-438d-8b49-bcf5db487478",
  "resumed_from": null,
  "turn_1": "Confirmed. If a claim record cannot be retrieved via domain tools, I will refuse to proceed rather than infer or fabricate any details. No invented records.",
  "turn_2": "You asked me to remember: **unknown claim records must be refused, not inferred.**",
  "turn_1_usage": {
    "input_tokens": 393,
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "output_tokens": 78,
    "server_tool_use": {
      "web_search_requests": 0,
      "web_fetch_requests": 0
    },
    "service_tier": "standard",
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "inference_geo": "global",
    "iterations": [
      {
        "input_tokens": 393,
        "output_tokens": 78,
        "cache_read_input_tokens": 0,
       

In [24]:
import asyncio


options = ClaudeAgentOptions(
    model=MODEL,
    tools=[],
    max_turns=MAX_TURNS,
    max_budget_usd=MAX_BUDGET_USD,
    cwd=NOTEBOOK_DIR,
    setting_sources=["project"],
)
started = time.perf_counter()


async def _sdk_run() -> dict[str, Any]:
    try:
        async with asyncio.timeout(TIMEOUT_SECONDS):
            async with ClaudeSDKClient(options=options) as client:
                await client.query(
                    "Explain in at most four bullets how turn limits, budget limits, "
                    "timeouts, and tool permissions bound an agent run."
                )
                result = await receive_result(client)
        return {
            "status": "sdk_error" if result.is_error else "success",
            "subtype": result.subtype,
            "session_id": result.session_id,
            "num_turns": result.num_turns,
            "stop_reason": result.stop_reason,
            "usage": result.usage,
            "model_usage": result.model_usage,
            "total_cost_usd": result.total_cost_usd,
            "budget_is_client_estimate": True,
            "configured_max_budget_usd": MAX_BUDGET_USD,
            "duration_ms": result.duration_ms,
            "outer_elapsed_seconds": round(time.perf_counter() - started, 3),
            "result": result.result,
        }
    except TimeoutError:
        return {
            "status": "application_timeout",
            "timeout_seconds": TIMEOUT_SECONDS,
            "outer_elapsed_seconds": round(time.perf_counter() - started, 3),
        }


usage_summary = run_sdk(_sdk_run)
print(json.dumps(usage_summary, default=str, indent=2))

{
  "status": "success",
  "subtype": "success",
  "session_id": "94bd3edb-8046-4025-89e0-5b7818aec404",
  "num_turns": 1,
  "stop_reason": "end_turn",
  "usage": {
    "input_tokens": 393,
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "output_tokens": 240,
    "server_tool_use": {
      "web_search_requests": 0,
      "web_fetch_requests": 0
    },
    "service_tier": "standard",
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "inference_geo": "global",
    "iterations": [
      {
        "input_tokens": 393,
        "output_tokens": 240,
        "cache_read_input_tokens": 0,
        "cache_creation_input_tokens": 0,
        "cache_creation": {
          "ephemeral_5m_input_tokens": 0,
          "ephemeral_1h_input_tokens": 0
        },
        "type": "message"
      }
    ],
    "speed": "standard"
  },
  "model_usage": {
    "claude-haiku-4-5-20251001": {
      "inputTokens": 541,
      "out

## 05 Permissions & model choice

Least privilege starts with `allowed_tools` and can be enforced dynamically with `can_use_tool`. The logistics exercise treats `release_hold` as a write: the callback and tool both require learner approval plus an idempotency key. The model comparison uses one short prompt with Sonnet and Opus; run it only if your account has access.

<img src="Images/d1pm_t5_provider_model_routing.png" width="850" alt="Route by task risk; authorize writes independently of model choice">

In [25]:
DEFAULT_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")
REASONING_MODEL = os.getenv("ANTHROPIC_REASONING_MODEL", "claude-opus-4-6")


async def short_model_run(model: str) -> dict[str, Any]:
    result: ResultMessage | None = None
    options = ClaudeAgentOptions(
        model=model,
        tools=[],
        max_turns=2,
        max_budget_usd=0.10,
        cwd=NOTEBOOK_DIR,
        setting_sources=["project"],
    )
    async for message in query(
        prompt="In one sentence, explain why a hold release needs approval and an idempotency key.",
        options=options,
    ):
        if isinstance(message, ResultMessage):
            result = message
    if result is None or result.is_error:
        raise RuntimeError(
            f"Model run failed for {model}; verify model access and SDK CLI runtime."
        )
    return {"model": model, "result": result.result, "total_cost_usd": result.total_cost_usd}


async def _sdk_run() -> dict[str, Any]:
    return {
        "default": await short_model_run(DEFAULT_MODEL),
        "reasoning": await short_model_run(REASONING_MODEL),
    }


print(json.dumps(run_sdk(_sdk_run), indent=2, default=str))

{
  "default": {
    "model": "claude-sonnet-4-6",
    "result": "A hold release is a **write operation** that modifies system state (freeing a shipment), so it requires **learner approval** to prevent unauthorized actions and an **idempotency key** to ensure the operation isn't accidentally executed more than once.",
    "total_cost_usd": 0.002646
  },
  "reasoning": {
    "model": "claude-opus-4-6",
    "result": "A hold release is a write operation that modifies shipment state, so it requires learner approval to prevent unauthorized changes and an idempotency key to ensure the operation is not accidentally executed more than once (e.g., due to retries or duplicate requests).",
    "total_cost_usd": 0.00524
  }
}


In [26]:
from pydantic import model_validator


class TriageOutcome(BaseModel):
    """Represent either an evidence-backed decision or a visible refusal."""

    status: Literal["completed", "refused"]
    decision: TriageDecision | None = None
    refusal_reason: str | None = None

    @model_validator(mode="after")
    def validate_outcome(self) -> "TriageOutcome":
        if self.status == "completed" and self.decision is None:
            raise ValueError("A completed outcome requires TriageDecision.")
        if self.status == "refused" and not self.refusal_reason:
            raise ValueError("A refused outcome requires refusal_reason.")
        return self


READ_TOOLS = {*INSURANCE_TOOL_NAMES, SHIPMENT_LOOKUP_TOOL, "StructuredOutput"}
trace: list[dict[str, object]] = []
permission_log: list[dict[str, object]] = []


async def permission_gate(
    tool_name: str,
    input_data: dict[str, Any],
    context: ToolPermissionContext,
) -> PermissionResultAllow | PermissionResultDeny:
    allowed = tool_name in READ_TOOLS
    event = {"event": "permission", "tool": tool_name, "allowed": allowed}
    permission_log.append(event)
    trace.append(event)
    if allowed:
        return PermissionResultAllow(updated_input=input_data)
    return PermissionResultDeny(
        message="Integrated preview is read-only; writes use the later deterministic boundary."
    )


async def _sdk_run() -> dict[str, Any]:
    options = ClaudeAgentOptions(
        model=MODEL,
        tools=[],
        mcp_servers={"insurance": insurance_server},
        can_use_tool=permission_gate,
        output_format={"type": "json_schema", "schema": TriageOutcome.model_json_schema()},
        max_turns=MAX_TURNS,
        max_budget_usd=MAX_BUDGET_USD,
        cwd=NOTEBOOK_DIR,
        setting_sources=["project"],
    )

    sdk_result: ResultMessage | None = None
    triage_tool_results: list[dict[str, object]] = []
    async with asyncio.timeout(TIMEOUT_SECONDS):
        async for message in query(
            prompt=prompt_stream(
                "Triage CLN-001. Call fnol_lookup first. For a found record, call claim_lookup "
                "and policy_lookup for its IDs and return status=completed with an evidence-backed "
                "TriageDecision using the CLM claim ID as case_id. If it is not found, return "
                "status=refused and do not invent a decision."
            ),
            options=options,
        ):
            if isinstance(message, SystemMessage):
                trace.append(
                    {"event": "system", "subtype": message.subtype, "data": message.data}
                )
            elif isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock):
                        trace.append(
                            {"event": "tool_use", "tool": block.name, "input": block.input}
                        )
            elif isinstance(message, UserMessage):
                for block in message.content:
                    if isinstance(block, ToolResultBlock):
                        evidence = {
                            "tool_use_id": block.tool_use_id,
                            "content": block.content,
                            "is_error": block.is_error,
                        }
                        triage_tool_results.append(evidence)
                        trace.append({"event": "tool_result", **evidence})
            elif isinstance(message, ResultMessage):
                sdk_result = message

    if sdk_result is None or sdk_result.is_error:
        raise RuntimeError("Integrated triage did not finish successfully.")
    triage_result = TriageOutcome.model_validate(sdk_result.structured_output)
    triage_evidence_text = json.dumps(triage_tool_results, default=str)
    if triage_result.decision and (
        not triage_result.decision.case_id.startswith("CLM-")
        or triage_result.decision.case_id not in triage_evidence_text
    ):
        raise ValueError(
            "Integrated triage case_id is not supported by captured tool evidence."
        )
    triage_session_id = sdk_result.session_id
    triage_usage = sdk_result.usage
    triage_cost_usd = sdk_result.total_cost_usd

    options = ClaudeAgentOptions(
        model=MODEL,
        tools=[],
        mcp_servers={"logistics": logistics_server},
        can_use_tool=permission_gate,
        max_turns=MAX_TURNS,
        max_budget_usd=MAX_BUDGET_USD,
        cwd=NOTEBOOK_DIR,
        setting_sources=["project"],
    )
    release_sdk_result: ResultMessage | None = None
    release_tool_results: list[dict[str, object]] = []
    async with asyncio.timeout(TIMEOUT_SECONDS):
        async for message in query(
            prompt=prompt_stream(
                "Call shipment_lookup for MSKU789 and report the latest status. Do not call "
                "release_hold and do not claim that any write occurred."
            ),
            options=options,
        ):
            if isinstance(message, UserMessage):
                for block in message.content:
                    if isinstance(block, ToolResultBlock):
                        evidence = {
                            "tool_use_id": block.tool_use_id,
                            "content": block.content,
                            "is_error": block.is_error,
                        }
                        release_tool_results.append(evidence)
                        trace.append({"event": "tool_result", **evidence})
            elif isinstance(message, ResultMessage):
                release_sdk_result = message

    if release_sdk_result is None or release_sdk_result.is_error:
        raise RuntimeError("Integrated shipment preview did not finish successfully.")
    release_records = shipment_records("MSKU789")
    if not release_records:
        raise ValueError("Shipment not found; no release proposal can be created.")
    latest_release_record = release_records[-1]
    if latest_release_record["event_id"] not in json.dumps(release_tool_results):
        raise ValueError("Integrated release proposal lacks captured shipment evidence.")
    release_preview_local = ReleaseProposal(
        container_id="MSKU789",
        source_event_id=latest_release_record["event_id"],
        should_release=latest_release_record["status"] == "held",
        reason=f"Latest authoritative shipment status is {latest_release_record['status']}.",
    )

    return {
        "triage": triage_result.model_dump(),
        "release_preview": release_preview_local.model_dump(),
        "sessions": {
            "triage": triage_session_id,
            "release_preview": release_sdk_result.session_id,
        },
        "usage": {
            "triage": triage_usage,
            "release_preview": release_sdk_result.usage,
        },
        "cost_usd": {
            "triage": triage_cost_usd,
            "release_preview": release_sdk_result.total_cost_usd,
        },
        "permission_log": permission_log,
        "trace": trace,
        "_release_preview_obj": release_preview_local,
    }


integrated_payload = run_sdk(_sdk_run)
release_preview = integrated_payload.pop("_release_preview_obj")
integrated_artifact = integrated_payload
print(json.dumps(integrated_artifact, default=str, indent=2))

{
  "triage": {
    "status": "completed",
    "decision": {
      "lane": "insurance",
      "case_id": "CLM-424063",
      "urgency": "high",
      "route_queue": "fnol_intake",
      "summary": "Jennifer Garcia (POL-787532354, active homeowners) reports vandalism/fence damage at 6650 Broadway. Claim CLM-424063 is open with a $5,834 reserve. Police report PR-20250822-4864 filed; photos available. Route immediately to fnol_intake for adjuster assignment.",
      "rationale": "1) FNOL CLN-001 confirmed via fnol_lookup \u2014 subject identifies CLM-424063 (property/fence damage, vandalism) under POL-787532354. 2) claim_lookup returns CLM-424063 as open, loss_type=property, urgency=high, reserve=$5,834, queue=fnol_intake. 3) policy_lookup confirms POL-787532354 is active, named insured=Jennifer Garcia, product=homeowners \u2014 coverage in force at time of loss. All three sources are consistent; no invented data. High urgency driven by vandalism event, police report on file, and signific

## 06 Cost awareness

Set `max_budget_usd` before every live run; it is a client-side estimate, not an invoice. Record `usage`, `model_usage`, and `total_cost_usd` from `ResultMessage`. The next cell writes a compact `usage_summary.json` under `W2D1/outputs/day1/`.

<img src="Images/d1pm_t6_cost_awareness.png" width="850" alt="Cost-aware bounded agent run with turn and spend caps">

In [27]:
# Usage telemetry is a local teaching artifact, not an operational write.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
usage_path = OUTPUT_DIR / "usage_summary.json"
usage_path.write_text(
    json.dumps(
        {
            "session_id": usage_summary.get("session_id"),
            "num_turns": usage_summary.get("num_turns"),
            "usage": usage_summary.get("usage"),
            "model_usage": usage_summary.get("model_usage"),
            "total_cost_usd": usage_summary.get("total_cost_usd"),
            "configured_max_budget_usd": MAX_BUDGET_USD,
            "budget_is_client_estimate": True,
        },
        default=str,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Wrote {usage_path}")

Wrote D:\Ambilio\EXL-CampusHire\Week2\W2D1\outputs\day1\usage_summary.json


In [28]:
# Sensitive write cell: all four conditions must be satisfied before the SDK is invoked.
write_ready = (
    ALLOW_LOCAL_WRITES
    and ALLOW_HOLD_RELEASE
    and LEARNER_APPROVED_HOLD_RELEASE
    and valid_idempotency_key(HOLD_RELEASE_IDEMPOTENCY_KEY)
)

if not write_ready:
    print(
        {
            "write_skipped": True,
            "required": [
                "ALLOW_LOCAL_WRITES=True",
                "ALLOW_HOLD_RELEASE=True",
                "LEARNER_APPROVED_HOLD_RELEASE=True",
                "a unique non-placeholder HOLD_RELEASE_IDEMPOTENCY_KEY (12+ characters)",
            ],
        }
    )
else:
    release_permission_log.clear()
    options = ClaudeAgentOptions(
        model=MODEL,
        tools=[],
        mcp_servers={"logistics": logistics_server},
        can_use_tool=release_permission_gate,
        max_turns=MAX_TURNS,
        max_budget_usd=MAX_BUDGET_USD,
        cwd=NOTEBOOK_DIR,
        setting_sources=["project"],
    )

    async def _sdk_run() -> dict[str, Any]:
        result: ResultMessage | None = None
        tool_results: list[dict[str, object]] = []
        async with asyncio.timeout(TIMEOUT_SECONDS):
            async for message in query(
                prompt=prompt_stream(
                    f"Call shipment_lookup for {release_preview.container_id}. If the "
                    "latest status remains held, call release_hold exactly once with "
                    f"container_id={release_preview.container_id!r}, approved=true, "
                    f"idempotency_key={HOLD_RELEASE_IDEMPOTENCY_KEY!r}, and "
                    f"source_event_id={release_preview.source_event_id!r}. Report the "
                    "tool result and never claim execution if it is denied."
                ),
                options=options,
            ):
                if isinstance(message, UserMessage):
                    for block in message.content:
                        if isinstance(block, ToolResultBlock):
                            tool_results.append(
                                {
                                    "tool_use_id": block.tool_use_id,
                                    "content": block.content,
                                    "is_error": block.is_error,
                                }
                            )
                elif isinstance(message, ResultMessage):
                    result = message

        if result is None or result.is_error:
            raise RuntimeError(
                "Permission-gated hold release did not finish successfully."
            )
        if not any(
            item["tool"] == RELEASE_HOLD_TOOL and item["allowed"]
            for item in release_permission_log
        ):
            raise RuntimeError(
                "The release tool was not authorized by the application callback."
            )
        return {
            "permission_log": release_permission_log,
            "tool_results": tool_results,
            "session_id": result.session_id,
            "usage": result.usage,
            "cost_usd": result.total_cost_usd,
        }

    print(json.dumps(run_sdk(_sdk_run), default=str, indent=2))

{'write_skipped': True, 'required': ['ALLOW_LOCAL_WRITES=True', 'ALLOW_HOLD_RELEASE=True', 'LEARNER_APPROVED_HOLD_RELEASE=True', 'a unique non-placeholder HOLD_RELEASE_IDEMPOTENCY_KEY (12+ characters)']}


## Exercises

Complete these after the live SDK sections. Use `run_sdk(...)` for any Claude Agent SDK call on Windows Jupyter. Solutions: [`_EXERCISES_SOLUTIONS.ipynb`](_EXERCISES_SOLUTIONS.ipynb).

### Exercise PM-1 — Allowlist only FNOL

**Concept:** `allowed_tools` is least privilege. If claim/policy tools are omitted, the agent cannot honestly complete evidence-backed triage.

**Task:** Rerun the grounded insurance triage pattern for **CLN-001**, but set `allowed_tools` to **only** the FNOL MCP tool name (one entry). Keep `mcp_servers={"insurance": insurance_server}`.

**Expected different result:** `called_tools` must not contain claim and policy lookups. Your post-checks that require all three insurance tools should fail (or you deliberately replace them with a narrower assertion that only FNOL ran).

**TODO hints (no code):**
- TODO: copy the §02 grounded triage options block
- TODO: replace `allowed_tools=INSURANCE_TOOL_NAMES` with a one-element list using the correct `mcp__insurance__...` FNOL name
- TODO: keep the same CLN-001 triage prompt
- TODO: print `called_tools` and show that claim/policy tools are absent
- TODO: decide whether to expect a validation `ValueError` or catch it and display `{"status": "incomplete_tools", ...}`

In [ ]:
# Exercise PM-1 workspace
# TODO: ClaudeAgentOptions with insurance_server and allowed_tools=[FNOL tool only]
# TODO: run_sdk + query triage for CLN-001; collect called_tools
# TODO: prove claim/policy tools were not called

raise NotImplementedError("Exercise PM-1: restrict allowed_tools to FNOL only")

### Exercise PM-2 — Tiny client budget

**Concept:** `max_budget_usd` is a client-side estimate that bounds a run; always inspect `ResultMessage` metering fields afterward.

**Task:** Run a **short** no-tools `query()` (reuse the §01 style prompt about grounded claims), but set `max_budget_usd=0.01` (much lower than the session default). Keep `tools=[]`.

**Expected different result:** compared with the default budget run, you should see a different cost/stop profile in `ResultMessage` (lower `total_cost_usd`, and/or an earlier stop / budget-related subtype depending on SDK version). Record whatever fields your SDK version exposes.

**TODO hints (no code):**
- TODO: build `ClaudeAgentOptions` with `tools=[]`, small `max_turns`, and `max_budget_usd=0.01`
- TODO: wrap the stream in `run_sdk`
- TODO: capture the terminal `ResultMessage`
- TODO: print `total_cost_usd`, `num_turns`, `stop_reason`, `subtype`, and `usage`
- TODO: write one sentence: did the tiny budget change behavior vs the earlier §01 run?

In [ ]:
# Exercise PM-2 workspace
# TODO: options with max_budget_usd=0.01 and tools=[]
# TODO: run_sdk query; print ResultMessage metering fields
# TODO: compare verbally or in JSON to the earlier default-budget §01 result

raise NotImplementedError("Exercise PM-2: run a tiny max_budget_usd query yourself")

## Next

Continue with [_ARCHITECTURE_COMPARISON.ipynb](_ARCHITECTURE_COMPARISON.ipynb). It is a separate lab, so this notebook does not duplicate it.

Exercise solutions (after you try): [_EXERCISES_SOLUTIONS.ipynb](_EXERCISES_SOLUTIONS.ipynb).